# 第 8 章 · 记忆（短期：SessionService）

`SessionService` 管**会话**（一次对话的线程 + 它的 state）。书里列了三种后端：

| 实现 | 存哪 | 适合 |
|---|---|---|
| `InMemorySessionService` | 进程内存 | 测试（重启即失） |
| `DatabaseSessionService` | 数据库（如 SQLite） | 要**跨进程持久化** |
| `VertexAiSessionService` | Vertex AI | GCP 生产环境 |

下面不只构造它们，还做三件事：

1. 把 4 条用户偏好各跑成一次会话，**验证 SQLite 那份真的持久化**（换新实例仍读得到）；
2. 看清 `SessionService` 的「检索」只有两种：按主键 `get_session`、按用户 `list_sessions`——
   **没有内容检索**。想按内容找，只能自己遍历全部会话的 `events`；
3. 打开库文件看实际落盘的行数。

第 2 点是这一节的重点：它解释了为什么还需要 `MemoryService`。

kernel 选 `Python (agentic-adk)`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async, require_adk)
quiet_adk_sync_warnings()

require_adk()      # 【适配】内核选错时给出可操作提示（内部就是 import google.adk）

import os
import sqlite3
import uuid

from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, DatabaseSessionService


# ---- ① 内存版 ----
mem_service = InMemorySessionService()
print("① InMemorySessionService 就绪（进程结束即失）")

# ---- ② SQLite 版：把 4 条用户偏好各跑成一次会话 ----
# 【适配】DatabaseSessionService 要求**异步**驱动，所以用 sqlite+aiosqlite（书里写的是 sqlite://）
DB_URL = "sqlite+aiosqlite:///./_session_demo.db"
DB_FILE = "_session_demo.db"
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)                      # 干净开始

APP, USER = "demo", "u1"
svc = DatabaseSessionService(db_url=DB_URL)

PREFERENCES = [
    "I am flying to New York next week and I prefer a window seat.",
    "I always book hotels near the city center, and I need a quiet room.",
    "I am vegetarian and I never eat spicy food.",
    "I prefer morning flights and I only travel with carry-on luggage.",
]
talk = LlmAgent(name="Talker", model=adk_model(), instruction="Answer in one short sentence.")
runner = Runner(app_name=APP, agent=talk, session_service=svc)   # 让 agent 把会话写进这个库

sids = []
for msg in PREFERENCES:
    sid = str(uuid.uuid4())
    run_async(svc.create_session(app_name=APP, user_id=USER, session_id=sid))
    for _ in retry_call(lambda: list(runner.run(
            user_id=USER, session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=msg)]))),
            label="run "):
        pass
    sids.append(sid)
print("② 已写入", len(sids), "个会话（4 条偏好各一次）")

# 相当于"重启"：换一个新实例读同一个库
svc2 = DatabaseSessionService(db_url=DB_URL)
s0 = run_async(svc2.get_session(app_name=APP, user_id=USER, session_id=sids[0]))
print("   新实例 svc2 按 session_id 读回来：events 有", len(s0.events), "条")

# ---- ③ SessionService 的"检索"就只有这两种 ----
# 没有 search：只能按主键精确取，或按用户列举。想按内容找，得自己遍历。
print("\n③ SessionService 与检索有关的方法：",
      [m for m in dir(svc2) if not m.startswith("_") and ("session" in m or "state" in m)])

alls = run_async(svc2.list_sessions(app_name=APP, user_id=USER)).sessions
print("   list_sessions 拿到", len(alls), "个会话（按最后更新时间排序）")

KEY = "vegetarian"
print(f"\n   想按内容找 {KEY!r}，只能自己遍历全部会话的 events：")
for meta in alls:
    full = run_async(svc2.get_session(app_name=APP, user_id=USER, session_id=meta.id))
    for ev in full.events:
        parts = ev.content.parts if ev.content else []
        txt = "".join(p.text for p in parts if getattr(p, "text", None))
        if KEY in txt.lower():
            print(f"     session {meta.id[:8]}…  author={ev.author:7}| {txt[:64]}")

# ---- ④ 库里实际存了什么 ----
con = sqlite3.connect(DB_FILE)
for t in ("sessions", "events", "app_states", "user_states"):
    print(f"   表 {t:12}: {con.execute(f'select count(*) from {t}').fetchone()[0]} 行")
con.close()

# ---- ⑤ Vertex AI 版 ----
try:
    from google.adk.sessions import VertexAiSessionService
    print("\n⑤ VertexAiSessionService 可导入，但需要 GCP 项目 + 认证，本机不实例化。")
except Exception as e:      # noqa: BLE001
    print("\n⑤ VertexAiSessionService 不可用:", type(e).__name__)


① InMemorySessionService 就绪（进程结束即失）


② 已写入 4 个会话（4 条偏好各一次）
   新实例 svc2 按 session_id 读回来：events 有 2 条

③ SessionService 与检索有关的方法： ['create_session', 'database_session_factory', 'delete_session', 'get_session', 'get_user_state', 'list_sessions']
   list_sessions 拿到 4 个会话（按最后更新时间排序）

   想按内容找 'vegetarian'，只能自己遍历全部会话的 events：
     session 39fdf3f0…  author=user   | I am vegetarian and I never eat spicy food.
     session 39fdf3f0…  author=Talker | I will make sure all my food recommendations for you are strictl
   表 sessions    : 4 行
   表 events      : 8 行
   表 app_states  : 1 行
   表 user_states : 1 行

⑤ VertexAiSessionService 可导入，但需要 GCP 项目 + 认证，本机不实例化。
